# 05 - Full System Backtest & Evaluation

This notebook provides comprehensive backtesting and evaluation of the complete 3-level hierarchical RL portfolio system.

## Evaluation Goals

1. **Performance Analysis**: Evaluate Meta Agent (full system) on test set
2. **Benchmark Comparison**: Compare against QQQ (Nasdaq-100 buy-and-hold)
3. **Transaction Cost Analysis**: Assess impact of portfolio turnover
4. **Risk Metrics**: Comprehensive risk-adjusted performance
5. **Portfolio Composition**: Analyze allocation patterns over time
6. **Final Report**: Production-ready performance summary

---

## Step 1: Setup & Imports

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Add helpers to path
sys.path.append('..')

from helpers import (
    create_meta_agent_env,
    create_super_agent_env,
    evaluate_meta_agent,
    calculate_all_metrics,
    plot_equity_curve,
    plot_drawdown,
    load_features_and_returns
)

from stable_baselines3 import PPO

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Imports successful")
print(f"Working directory: {os.getcwd()}")
print(f"Evaluation date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 2: Load All Trained Models & Results

In [ ]:
# Define paths
MODELS_DIR = Path('../models')
DATA_DIR = Path('../data_hierarchical')

# Load metadata
with open(DATA_DIR / 'metadata.json', 'r') as f:
    metadata = json.load(f)

print("Portfolio Configuration:")
print(f"  Tickers: {', '.join(metadata['tickers'])}")
print(f"  Benchmark: {metadata['benchmark']}")
print(f"  Total Weeks: {metadata['date_range']['total_weeks']}")
print(f"  Date Range: {metadata['date_range']['start']} to {metadata['date_range']['end']}")

print(f"\nTest Set:")
print(f"  Period: {metadata['splits']['test']['start']} to {metadata['splits']['test']['end']}")
print(f"  Weeks: {metadata['splits']['test']['weeks']}")

# Load all agent results
print("\nLoading trained agent results...")

with open(MODELS_DIR / 'agent_models' / 'technical_results.json', 'r') as f:
    tech_results = json.load(f)

with open(MODELS_DIR / 'agent_models' / 'sentiment_results.json', 'r') as f:
    sent_results = json.load(f)

with open(MODELS_DIR / 'super_agent_results.json', 'r') as f:
    super_results = json.load(f)

with open(MODELS_DIR / 'meta_agent_results.json', 'r') as f:
    meta_results = json.load(f)

print("✓ All agent results loaded")

# Load Meta Agent model
meta_agent_path = MODELS_DIR / 'meta_agent_production.zip'
print(f"\nLoading Meta Agent model from: {meta_agent_path}")
meta_model = PPO.load(meta_agent_path)
print("✓ Meta Agent model loaded")

## Step 3: Load Benchmark Data (QQQ)

We'll compare our hierarchical RL system against a simple buy-and-hold strategy in QQQ.

In [ ]:
import yfinance as yf

print("Downloading QQQ benchmark data...")

# Download QQQ data for the full period
start_date = metadata['date_range']['start']
end_date = metadata['date_range']['end']

qqq_data = yf.download('QQQ', start=start_date, end=end_date, progress=False)

# Handle column structure
if isinstance(qqq_data.columns, pd.MultiIndex):
    qqq_data.columns = qqq_data.columns.get_level_values(0)

qqq_data.columns = [str(col).lower().replace(' ', '_') for col in qqq_data.columns]
adj_close_col = 'adj_close' if 'adj_close' in qqq_data.columns else 'close'

# Resample to weekly (Friday close)
qqq_weekly = qqq_data[adj_close_col].resample('W-FRI').last().ffill()

# Calculate weekly returns
qqq_returns = qqq_weekly.pct_change().dropna()

# Get test period returns
test_start = metadata['splits']['test']['start']
test_end = metadata['splits']['test']['end']
qqq_test_returns = qqq_returns.loc[test_start:test_end]

print(f"✓ QQQ benchmark data loaded")
print(f"  Test period weeks: {len(qqq_test_returns)}")
print(f"  Date range: {qqq_test_returns.index[0].strftime('%Y-%m-%d')} to {qqq_test_returns.index[-1].strftime('%Y-%m-%d')}")

## Step 4: Calculate Benchmark Metrics

In [ ]:
# Calculate QQQ benchmark metrics
qqq_metrics = calculate_all_metrics(qqq_test_returns.values)

print("QQQ Benchmark Performance (Test Set):")
print("="*60)
print(f"  Total Return: {qqq_metrics['total_return']:.2%}")
print(f"  Annual Return: {qqq_metrics['annual_return']:.2%}")
print(f"  Annual Volatility: {qqq_metrics['annual_volatility']:.2%}")
print(f"  Sharpe Ratio: {qqq_metrics['sharpe_ratio']:.3f}")
print(f"  Sortino Ratio: {qqq_metrics['sortino_ratio']:.3f}")
print(f"  Max Drawdown: {qqq_metrics['max_drawdown']:.2%}")
print(f"  Calmar Ratio: {qqq_metrics['calmar_ratio']:.3f}")
print(f"  Win Rate: {qqq_metrics['win_rate']:.2%}")

## Step 5: Hierarchical System vs Benchmark Comparison

In [ ]:
# Create comparison table
test_meta = meta_results['test']

comparison_table = pd.DataFrame([
    {
        'Strategy': 'QQQ (Buy & Hold)',
        'Total Return': qqq_metrics['total_return'],
        'Annual Return': qqq_metrics['annual_return'],
        'Volatility': qqq_metrics['annual_volatility'],
        'Sharpe': qqq_metrics['sharpe_ratio'],
        'Sortino': qqq_metrics['sortino_ratio'],
        'Max DD': qqq_metrics['max_drawdown'],
        'Calmar': qqq_metrics['calmar_ratio'],
        'Win Rate': qqq_metrics['win_rate']
    },
    {
        'Strategy': 'Technical Agent (L1)',
        'Total Return': (1 + pd.Series(tech_results['test']['returns'])).prod() - 1,
        'Annual Return': tech_results['test']['annual_return'],
        'Volatility': tech_results['test']['annual_volatility'],
        'Sharpe': tech_results['test']['sharpe'],
        'Sortino': tech_results['test'].get('sortino', np.nan),
        'Max DD': tech_results['test']['max_drawdown'],
        'Calmar': tech_results['test']['calmar'],
        'Win Rate': tech_results['test']['win_rate']
    },
    {
        'Strategy': 'Sentiment Agent (L1)',
        'Total Return': (1 + pd.Series(sent_results['test']['returns'])).prod() - 1,
        'Annual Return': sent_results['test']['annual_return'],
        'Volatility': sent_results['test']['annual_volatility'],
        'Sharpe': sent_results['test']['sharpe'],
        'Sortino': sent_results['test'].get('sortino', np.nan),
        'Max DD': sent_results['test']['max_drawdown'],
        'Calmar': sent_results['test']['calmar'],
        'Win Rate': sent_results['test']['win_rate']
    },
    {
        'Strategy': 'Super Agent (L2)',
        'Total Return': (1 + pd.Series(super_results['test']['returns'])).prod() - 1,
        'Annual Return': super_results['test']['annual_return'],
        'Volatility': super_results['test']['annual_volatility'],
        'Sharpe': super_results['test']['sharpe'],
        'Sortino': super_results['test'].get('sortino', np.nan),
        'Max DD': super_results['test']['max_drawdown'],
        'Calmar': super_results['test']['calmar'],
        'Win Rate': super_results['test']['win_rate']
    },
    {
        'Strategy': 'Meta Agent (L3) ★',
        'Total Return': (1 + pd.Series(test_meta['returns'])).prod() - 1,
        'Annual Return': test_meta['annual_return'],
        'Volatility': test_meta['annual_volatility'],
        'Sharpe': test_meta['sharpe'],
        'Sortino': test_meta.get('sortino', np.nan),
        'Max DD': test_meta['max_drawdown'],
        'Calmar': test_meta['calmar'],
        'Win Rate': test_meta['win_rate']
    }
])

print("\n" + "="*100)
print("COMPREHENSIVE PERFORMANCE COMPARISON (Test Set)")
print("="*100)
print(comparison_table.to_string(index=False))

# Calculate outperformance
print("\n" + "="*100)
print("META AGENT vs BENCHMARK")
print("="*100)

sharpe_improvement = (test_meta['sharpe'] - qqq_metrics['sharpe_ratio']) / qqq_metrics['sharpe_ratio'] * 100
return_improvement = (test_meta['annual_return'] - qqq_metrics['annual_return']) / qqq_metrics['annual_return'] * 100
dd_improvement = (test_meta['max_drawdown'] - qqq_metrics['max_drawdown']) / abs(qqq_metrics['max_drawdown']) * 100

print(f"\nSharpe Ratio:")
print(f"  Meta Agent: {test_meta['sharpe']:.3f}")
print(f"  QQQ Benchmark: {qqq_metrics['sharpe_ratio']:.3f}")
print(f"  Improvement: {sharpe_improvement:+.1f}%")

print(f"\nAnnual Return:")
print(f"  Meta Agent: {test_meta['annual_return']:.2%}")
print(f"  QQQ Benchmark: {qqq_metrics['annual_return']:.2%}")
print(f"  Difference: {test_meta['annual_return'] - qqq_metrics['annual_return']:+.2%}")

print(f"\nMax Drawdown:")
print(f"  Meta Agent: {test_meta['max_drawdown']:.2%}")
print(f"  QQQ Benchmark: {qqq_metrics['max_drawdown']:.2%}")
print(f"  {'Better' if test_meta['max_drawdown'] > qqq_metrics['max_drawdown'] else 'Worse'} by {abs(dd_improvement):.1f}%")

print(f"\nCalmar Ratio (Return/Drawdown):")
print(f"  Meta Agent: {test_meta['calmar']:.3f}")
print(f"  QQQ Benchmark: {qqq_metrics['calmar_ratio']:.3f}")

## Step 6: Comprehensive Visualization

In [ ]:
# Create comprehensive performance visualization
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Cumulative Returns (large plot)
ax1 = fig.add_subplot(gs[0, :])
qqq_cumulative = (1 + qqq_test_returns).cumprod()
tech_cumulative = (1 + pd.Series(tech_results['test']['returns'])).cumprod()
sent_cumulative = (1 + pd.Series(sent_results['test']['returns'])).cumprod()
super_cumulative = (1 + pd.Series(super_results['test']['returns'])).cumprod()
meta_cumulative = (1 + pd.Series(test_meta['returns'])).cumprod()

ax1.plot(qqq_cumulative.values, label=f'QQQ (Sharpe: {qqq_metrics["sharpe_ratio"]:.2f})', 
         linewidth=2, alpha=0.7, linestyle='--')
ax1.plot(tech_cumulative.values, label=f'Technical L1 (Sharpe: {tech_results["test"]["sharpe"]:.2f})', 
         alpha=0.4, linewidth=1)
ax1.plot(sent_cumulative.values, label=f'Sentiment L1 (Sharpe: {sent_results["test"]["sharpe"]:.2f})', 
         alpha=0.4, linewidth=1)
ax1.plot(super_cumulative.values, label=f'Super L2 (Sharpe: {super_results["test"]["sharpe"]:.2f})', 
         alpha=0.6, linewidth=1.5)
ax1.plot(meta_cumulative.values, label=f'Meta L3 (Sharpe: {test_meta["sharpe"]:.2f})', 
         linewidth=3)
ax1.set_xlabel('Week', fontsize=11)
ax1.set_ylabel('Cumulative Return', fontsize=11)
ax1.set_title('Test Set: Cumulative Returns Comparison', fontsize=13, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# 2. Drawdown Comparison
ax2 = fig.add_subplot(gs[1, 0])
qqq_dd = pd.Series(qqq_test_returns.values).cumsum()
qqq_dd = qqq_dd - qqq_dd.cummax()
meta_dd = pd.Series(test_meta['returns']).cumsum()
meta_dd = meta_dd - meta_dd.cummax()

ax2.fill_between(range(len(qqq_dd)), qqq_dd, 0, alpha=0.3, label='QQQ', color='orange')
ax2.fill_between(range(len(meta_dd)), meta_dd, 0, alpha=0.5, label='Meta Agent', color='blue')
ax2.set_xlabel('Week')
ax2.set_ylabel('Drawdown')
ax2.set_title('Drawdown Comparison')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# 3. Return Distribution
ax3 = fig.add_subplot(gs[1, 1])
ax3.hist(qqq_test_returns.values, bins=20, alpha=0.5, label='QQQ', density=True, color='orange')
ax3.hist(test_meta['returns'], bins=20, alpha=0.5, label='Meta Agent', density=True, color='blue')
ax3.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax3.axvline(x=np.mean(qqq_test_returns.values), color='orange', linestyle=':', alpha=0.7, label='QQQ Mean')
ax3.axvline(x=np.mean(test_meta['returns']), color='blue', linestyle=':', alpha=0.7, label='Meta Mean')
ax3.set_xlabel('Weekly Return')
ax3.set_ylabel('Density')
ax3.set_title('Return Distribution')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# 4. Risk-Return Scatter
ax4 = fig.add_subplot(gs[1, 2])
strategies = ['QQQ', 'Tech', 'Sent', 'Super', 'Meta']
returns = [
    qqq_metrics['annual_return'],
    tech_results['test']['annual_return'],
    sent_results['test']['annual_return'],
    super_results['test']['annual_return'],
    test_meta['annual_return']
]
vols = [
    qqq_metrics['annual_volatility'],
    tech_results['test']['annual_volatility'],
    sent_results['test']['annual_volatility'],
    super_results['test']['annual_volatility'],
    test_meta['annual_volatility']
]
colors = ['orange', 'blue', 'green', 'purple', 'red']
sizes = [150, 100, 100, 120, 200]

for i, (strat, ret, vol, color, size) in enumerate(zip(strategies, returns, vols, colors, sizes)):
    ax4.scatter(vol, ret, s=size, alpha=0.6, c=color, label=strat, edgecolors='black', linewidth=1.5 if strat=='Meta' else 0.5)
    ax4.annotate(strat, (vol, ret), fontsize=9, ha='center', fontweight='bold' if strat=='Meta' else 'normal')

ax4.set_xlabel('Annual Volatility')
ax4.set_ylabel('Annual Return')
ax4.set_title('Risk-Return Profile')
ax4.grid(True, alpha=0.3)

# 5. Sharpe Ratio Comparison
ax5 = fig.add_subplot(gs[2, 0])
sharpes = [
    qqq_metrics['sharpe_ratio'],
    tech_results['test']['sharpe'],
    sent_results['test']['sharpe'],
    super_results['test']['sharpe'],
    test_meta['sharpe']
]
bars = ax5.bar(strategies, sharpes, color=colors, alpha=0.7, edgecolor='black')
bars[-1].set_linewidth(2.5)  # Highlight Meta Agent
ax5.axhline(y=2.0, color='red', linestyle='--', alpha=0.5, label='Target (2.0)')
ax5.set_ylabel('Sharpe Ratio')
ax5.set_title('Sharpe Ratio Comparison')
ax5.legend()
ax5.grid(True, alpha=0.3, axis='y')

# 6. Calmar Ratio Comparison
ax6 = fig.add_subplot(gs[2, 1])
calmars = [
    qqq_metrics['calmar_ratio'],
    tech_results['test']['calmar'],
    sent_results['test']['calmar'],
    super_results['test']['calmar'],
    test_meta['calmar']
]
bars = ax6.bar(strategies, calmars, color=colors, alpha=0.7, edgecolor='black')
bars[-1].set_linewidth(2.5)
ax6.set_ylabel('Calmar Ratio')
ax6.set_title('Calmar Ratio (Return/Drawdown)')
ax6.grid(True, alpha=0.3, axis='y')

# 7. Win Rate Comparison
ax7 = fig.add_subplot(gs[2, 2])
win_rates = [
    qqq_metrics['win_rate'],
    tech_results['test']['win_rate'],
    sent_results['test']['win_rate'],
    super_results['test']['win_rate'],
    test_meta['win_rate']
]
bars = ax7.bar(strategies, win_rates, color=colors, alpha=0.7, edgecolor='black')
bars[-1].set_linewidth(2.5)
ax7.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50%')
ax7.set_ylabel('Win Rate')
ax7.set_title('Win Rate')
ax7.set_ylim([0, 1])
ax7.legend()
ax7.grid(True, alpha=0.3, axis='y')

plt.suptitle('Multi-Hierarchical RL Portfolio System - Comprehensive Backtest', 
             fontsize=15, fontweight='bold', y=0.995)

plt.show()

## Step 7: Transaction Cost Analysis

Analyze portfolio turnover and its impact on net returns.

In [ ]:
# Recreate test environment to get portfolio weights
print("Analyzing transaction costs and turnover...\n")

technical_model_path = MODELS_DIR / 'agent_models' / 'technical_ema_sharpe.zip'
sentiment_model_path = MODELS_DIR / 'agent_models' / 'sentiment_ema_sharpe.zip'
super_agent_path = MODELS_DIR / 'super_agent_production.zip'

# Create test environment
super_env_test = create_super_agent_env(
    technical_model_path=str(technical_model_path),
    sentiment_model_path=str(sentiment_model_path),
    split='test',
    reward_type='ema_sharpe'
)

test_env = create_meta_agent_env(
    super_agent_path=str(super_agent_path),
    super_env=super_env_test,
    split='test',
    reward_type='ema_sharpe'
)

# Run episode to collect portfolio weights
obs, _ = test_env.reset()
done = False
portfolio_weights = []
episode_returns = []

while not done:
    action, _ = meta_model.predict(obs, deterministic=True)
    # Normalize action to get portfolio weights
    weights = action / action.sum()
    portfolio_weights.append(weights)
    
    obs, reward, done, truncated, info = test_env.step(action)
    episode_returns.append(info.get('return', 0))
    done = done or truncated

portfolio_weights = np.array(portfolio_weights)

# Calculate turnover
turnover = np.abs(portfolio_weights[1:] - portfolio_weights[:-1]).sum(axis=1)
avg_turnover = turnover.mean()
total_turnover = turnover.sum()

# Calculate transaction costs (0.1% = 0.001)
transaction_cost_rate = 0.001
transaction_costs = turnover * transaction_cost_rate
total_tc = transaction_costs.sum()

# Calculate net returns after transaction costs
gross_returns = np.array(test_meta['returns'])
tc_per_period = np.concatenate([[0], transaction_costs])  # No TC on first period
net_returns = gross_returns - tc_per_period

# Recalculate metrics with transaction costs
net_metrics = calculate_all_metrics(net_returns)

print("Transaction Cost Analysis:")
print("="*60)
print(f"\nPortfolio Turnover:")
print(f"  Average Weekly Turnover: {avg_turnover:.2%}")
print(f"  Total Turnover (Test Period): {total_turnover:.2%}")
print(f"  Annualized Turnover: {avg_turnover * 52:.2%}")

print(f"\nTransaction Costs (0.1% per trade):")
print(f"  Total TC (Test Period): {total_tc:.4f} ({total_tc*100:.2f}%)")
print(f"  Average TC per Week: {transaction_costs.mean():.6f}")
print(f"  Max TC in Single Week: {transaction_costs.max():.6f}")

print(f"\nPerformance Impact:")
print(f"  Gross Sharpe: {test_meta['sharpe']:.3f}")
print(f"  Net Sharpe (after TC): {net_metrics['sharpe_ratio']:.3f}")
print(f"  Sharpe Degradation: {(test_meta['sharpe'] - net_metrics['sharpe_ratio']):.3f}")

print(f"\n  Gross Annual Return: {test_meta['annual_return']:.2%}")
print(f"  Net Annual Return (after TC): {net_metrics['annual_return']:.2%}")
print(f"  Return Degradation: {(test_meta['annual_return'] - net_metrics['annual_return']):.2%}")

# Visualize turnover
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Turnover over time
axes[0].plot(turnover, alpha=0.7)
axes[0].axhline(y=avg_turnover, color='r', linestyle='--', alpha=0.5, label=f'Avg: {avg_turnover:.2%}')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Portfolio Turnover')
axes[0].set_title('Portfolio Turnover Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative transaction costs
cumulative_tc = np.cumsum(transaction_costs)
axes[1].plot(cumulative_tc, color='red', alpha=0.7)
axes[1].fill_between(range(len(cumulative_tc)), cumulative_tc, 0, alpha=0.3, color='red')
axes[1].set_xlabel('Week')
axes[1].set_ylabel('Cumulative Transaction Costs')
axes[1].set_title(f'Cumulative TC: {total_tc:.4f} ({total_tc*100:.2f}%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Portfolio Composition Analysis

In [ ]:
# Analyze portfolio composition over time
tickers = metadata['tickers']

# Create dataframe of portfolio weights
weights_df = pd.DataFrame(portfolio_weights, columns=tickers)

# Calculate average weights
avg_weights = weights_df.mean()
print("Average Portfolio Allocation:")
print("="*60)
for ticker, weight in avg_weights.sort_values(ascending=False).items():
    print(f"  {ticker}: {weight:.2%}")

# Visualize portfolio evolution
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Stacked area chart
axes[0].stackplot(range(len(weights_df)), *[weights_df[ticker].values for ticker in tickers],
                   labels=tickers, alpha=0.7)
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Portfolio Weight')
axes[0].set_title('Portfolio Composition Over Time (Stacked)')
axes[0].legend(loc='upper left', ncol=len(tickers), fontsize=9)
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3)

# Individual weight evolution
for ticker in tickers:
    axes[1].plot(weights_df[ticker], label=ticker, alpha=0.7, linewidth=1.5)

axes[1].set_xlabel('Week')
axes[1].set_ylabel('Weight')
axes[1].set_title('Individual Asset Weights Over Time')
axes[1].legend(loc='upper left', ncol=len(tickers), fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Weight statistics
print("\nPortfolio Weight Statistics:")
print("="*60)
print(weights_df.describe())

## Step 9: Final Performance Report

In [ ]:
# Generate comprehensive final report
print("\n" + "="*80)
print("FINAL PERFORMANCE REPORT")
print("Multi-Hierarchical RL Portfolio Optimization System")
print("="*80)

print(f"\nEvaluation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Test Period: {metadata['splits']['test']['start']} to {metadata['splits']['test']['end']}")
print(f"Test Duration: {metadata['splits']['test']['weeks']} weeks")
print(f"Assets: {', '.join(tickers)}")
print(f"Benchmark: {metadata['benchmark']} (Nasdaq-100)")

print("\n" + "-"*80)
print("HIERARCHICAL SYSTEM PERFORMANCE")
print("-"*80)

print(f"\nLevel 1 - Base Agents:")
print(f"  Technical Agent Sharpe: {tech_results['test']['sharpe']:.3f}")
print(f"  Sentiment Agent Sharpe: {sent_results['test']['sharpe']:.3f}")

print(f"\nLevel 2 - Super Agent (Blending):")
print(f"  Sharpe: {super_results['test']['sharpe']:.3f}")
print(f"  Improvement over best base: {(super_results['test']['sharpe'] - max(tech_results['test']['sharpe'], sent_results['test']['sharpe'])) / max(tech_results['test']['sharpe'], sent_results['test']['sharpe']) * 100:+.1f}%")

print(f"\nLevel 3 - Meta Agent (Macro-Aware):")
print(f"  Gross Sharpe: {test_meta['sharpe']:.3f}")
print(f"  Net Sharpe (after TC): {net_metrics['sharpe_ratio']:.3f}")
print(f"  Improvement over Super: {(test_meta['sharpe'] - super_results['test']['sharpe']) / super_results['test']['sharpe'] * 100:+.1f}%")

print("\n" + "-"*80)
print("META AGENT vs BENCHMARK (QQQ)")
print("-"*80)

comparison_metrics = [
    ('Total Return', f"{net_metrics['total_return']:.2%}", f"{qqq_metrics['total_return']:.2%}"),
    ('Annual Return', f"{net_metrics['annual_return']:.2%}", f"{qqq_metrics['annual_return']:.2%}"),
    ('Annual Volatility', f"{net_metrics['annual_volatility']:.2%}", f"{qqq_metrics['annual_volatility']:.2%}"),
    ('Sharpe Ratio', f"{net_metrics['sharpe_ratio']:.3f}", f"{qqq_metrics['sharpe_ratio']:.3f}"),
    ('Sortino Ratio', f"{net_metrics['sortino_ratio']:.3f}", f"{qqq_metrics['sortino_ratio']:.3f}"),
    ('Max Drawdown', f"{net_metrics['max_drawdown']:.2%}", f"{qqq_metrics['max_drawdown']:.2%}"),
    ('Calmar Ratio', f"{net_metrics['calmar_ratio']:.3f}", f"{qqq_metrics['calmar_ratio']:.3f}"),
    ('Win Rate', f"{net_metrics['win_rate']:.2%}", f"{qqq_metrics['win_rate']:.2%}")
]

print(f"\n{'Metric':<20} {'Meta Agent':<15} {'QQQ Benchmark':<15} {'Difference':<15}")
print("-"*80)
for metric, meta_val, qqq_val in comparison_metrics:
    # Calculate difference
    meta_num = float(meta_val.replace('%', '').replace(',', ''))
    qqq_num = float(qqq_val.replace('%', '').replace(',', ''))
    diff = meta_num - qqq_num
    
    if '%' in meta_val:
        diff_str = f"{diff:+.2f}%"
    else:
        diff_str = f"{diff:+.3f}"
    
    print(f"{metric:<20} {meta_val:<15} {qqq_val:<15} {diff_str:<15}")

print("\n" + "-"*80)
print("TRANSACTION COST IMPACT")
print("-"*80)
print(f"\nAverage Weekly Turnover: {avg_turnover:.2%}")
print(f"Annualized Turnover: {avg_turnover * 52:.2%}")
print(f"Total Transaction Costs: {total_tc*100:.2f}%")
print(f"Sharpe Degradation from TC: {(test_meta['sharpe'] - net_metrics['sharpe_ratio']):.3f}")
print(f"Return Degradation from TC: {(test_meta['annual_return'] - net_metrics['annual_return']):.2%}")

print("\n" + "-"*80)
print("PORTFOLIO COMPOSITION")
print("-"*80)
print(f"\nAverage Allocation:")
for ticker, weight in avg_weights.sort_values(ascending=False).items():
    print(f"  {ticker}: {weight:.2%}")

# Calculate concentration metrics
herfindahl_index = (avg_weights ** 2).sum()
effective_n = 1 / herfindahl_index

print(f"\nConcentration Metrics:")
print(f"  Herfindahl Index: {herfindahl_index:.3f}")
print(f"  Effective Number of Assets: {effective_n:.2f} (out of {len(tickers)})")

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)

# Determine performance verdict
if net_metrics['sharpe_ratio'] > qqq_metrics['sharpe_ratio']:
    verdict = "OUTPERFORMS"
    performance_pct = (net_metrics['sharpe_ratio'] - qqq_metrics['sharpe_ratio']) / qqq_metrics['sharpe_ratio'] * 100
    print(f"\n✓ The Meta Agent {verdict} the QQQ benchmark by {performance_pct:.1f}% (Sharpe basis)")
else:
    verdict = "UNDERPERFORMS"
    performance_pct = (qqq_metrics['sharpe_ratio'] - net_metrics['sharpe_ratio']) / qqq_metrics['sharpe_ratio'] * 100
    print(f"\n✗ The Meta Agent {verdict} the QQQ benchmark by {performance_pct:.1f}% (Sharpe basis)")

if net_metrics['sharpe_ratio'] >= 2.0:
    print(f"✓ Target Sharpe ratio of 2.0+ ACHIEVED: {net_metrics['sharpe_ratio']:.3f}")
else:
    print(f"✗ Target Sharpe ratio of 2.0 not reached: {net_metrics['sharpe_ratio']:.3f}")

if abs(net_metrics['max_drawdown']) < abs(qqq_metrics['max_drawdown']):
    print(f"✓ Lower drawdown than benchmark: {net_metrics['max_drawdown']:.2%} vs {qqq_metrics['max_drawdown']:.2%}")
else:
    print(f"✗ Higher drawdown than benchmark: {net_metrics['max_drawdown']:.2%} vs {qqq_metrics['max_drawdown']:.2%}")

print("\n" + "="*80)
print(f"Report generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

## Step 10: Save Final Results

In [ ]:
# Save comprehensive backtest results
backtest_results = {
    'evaluation_date': datetime.now().isoformat(),
    'test_period': {
        'start': metadata['splits']['test']['start'],
        'end': metadata['splits']['test']['end'],
        'weeks': metadata['splits']['test']['weeks']
    },
    'meta_agent': {
        'gross': {
            'sharpe': float(test_meta['sharpe']),
            'annual_return': float(test_meta['annual_return']),
            'annual_volatility': float(test_meta['annual_volatility']),
            'max_drawdown': float(test_meta['max_drawdown']),
            'calmar': float(test_meta['calmar']),
            'win_rate': float(test_meta['win_rate'])
        },
        'net': {
            'sharpe': float(net_metrics['sharpe_ratio']),
            'annual_return': float(net_metrics['annual_return']),
            'annual_volatility': float(net_metrics['annual_volatility']),
            'max_drawdown': float(net_metrics['max_drawdown']),
            'calmar': float(net_metrics['calmar_ratio']),
            'win_rate': float(net_metrics['win_rate'])
        }
    },
    'benchmark_qqq': {
        'sharpe': float(qqq_metrics['sharpe_ratio']),
        'annual_return': float(qqq_metrics['annual_return']),
        'annual_volatility': float(qqq_metrics['annual_volatility']),
        'max_drawdown': float(qqq_metrics['max_drawdown']),
        'calmar': float(qqq_metrics['calmar_ratio']),
        'win_rate': float(qqq_metrics['win_rate'])
    },
    'transaction_costs': {
        'avg_weekly_turnover': float(avg_turnover),
        'annualized_turnover': float(avg_turnover * 52),
        'total_tc_pct': float(total_tc * 100),
        'sharpe_degradation': float(test_meta['sharpe'] - net_metrics['sharpe_ratio']),
        'return_degradation_pct': float((test_meta['annual_return'] - net_metrics['annual_return']) * 100)
    },
    'portfolio_composition': {
        'avg_weights': {ticker: float(weight) for ticker, weight in avg_weights.items()},
        'herfindahl_index': float(herfindahl_index),
        'effective_n_assets': float(effective_n)
    },
    'outperformance': {
        'sharpe_improvement_pct': float((net_metrics['sharpe_ratio'] - qqq_metrics['sharpe_ratio']) / qqq_metrics['sharpe_ratio'] * 100),
        'return_difference_pct': float((net_metrics['annual_return'] - qqq_metrics['annual_return']) * 100),
        'verdict': verdict
    }
}

with open('../models/final_backtest_results.json', 'w') as f:
    json.dump(backtest_results, f, indent=2)

print("✓ Final backtest results saved to models/final_backtest_results.json")

## Summary

### What We Accomplished

1. ✅ Comprehensive backtesting of full hierarchical system
2. ✅ Benchmark comparison against QQQ (Nasdaq-100)
3. ✅ Transaction cost analysis and turnover evaluation
4. ✅ Portfolio composition analysis over time
5. ✅ Complete performance report with all metrics
6. ✅ Detailed visualizations of all aspects
7. ✅ Final results saved for production use

### Key Takeaways

**System Performance**:
- Check final Sharpe ratio vs benchmark above
- Review risk-adjusted returns and drawdowns
- Analyze transaction cost impact on net performance

**Hierarchical Value-Add**:
- Level 1 (Base Agents): Capture technical & sentiment signals
- Level 2 (Super Agent): Optimal blending of complementary strategies
- Level 3 (Meta Agent): Macro-aware regime adaptation

**Portfolio Insights**:
- Check average allocation across 7 tech stocks
- Review concentration metrics (Herfindahl index)
- Analyze rebalancing frequency and turnover

---

## 🎉 PROJECT COMPLETE!

You have successfully:
1. ✅ Prepared multi-modal financial data (44 features)
2. ✅ Trained 2 specialized base agents (Technical & Sentiment)
3. ✅ Created Super Agent to blend base strategies
4. ✅ Developed Meta Agent for macro-aware allocation
5. ✅ Backtested full system with comprehensive evaluation

### Next Steps (Optional Enhancements)

- **Walk-Forward Analysis**: Retrain agents periodically on rolling windows
- **Risk Constraints**: Add VaR/CVaR limits to environment
- **Alternative Algorithms**: Try SAC, TD3, or A2C
- **More Hierarchies**: Add sector-level agents
- **Live Trading**: Deploy with paper trading API
- **Ensemble Methods**: Combine multiple trained models

### Files Generated

- `models/agent_models/technical_ema_sharpe.zip` - Base agent
- `models/agent_models/sentiment_ema_sharpe.zip` - Base agent
- `models/super_agent_production.zip` - Level 2 agent
- `models/meta_agent_production.zip` - Level 3 agent (FINAL)
- `models/final_backtest_results.json` - Complete evaluation

---

**Thank you for using the Multi-Hierarchical RL Portfolio System!**